# OpenLithoHub — Quickstart in Colab

Three minutes from "empty notebook" to "paper-ready figure". This notebook deliberately avoids the heavy `[workflow]` extras (KLayout, scipy) so it runs cleanly on Colab's stock runtime.

What you'll do:
1. Install OpenLithoHub with the `jupyter` extra.
2. Generate a dummy layout — no dataset download required.
3. Score it against a slightly perturbed "prediction" with the EPE and PV-band metrics.
4. Render an IEEE-style contour figure.

Run cells top-to-bottom with **Shift+Enter**.

## 1. Install

In [ ]:
!pip install --quiet 'openlithohub[jupyter]'

## 2. Generate a dummy layout

`generate_dummy_layout` returns a binary `torch.Tensor` that already satisfies basic min-width / min-spacing rules for the configured pixel pitch.

In [ ]:
from openlithohub.data import generate_dummy_layout

target = generate_dummy_layout(size=256, seed=0)
print("shape:", tuple(target.shape), "fill:", float(target.mean()))

## 3. Score a perturbed prediction

We simulate an OPC "prediction" by perturbing the target. In a real workflow this would come from your model.

In [ ]:
from openlithohub._utils.morphology import binary_dilation, binary_erosion
from openlithohub.benchmark.metrics.epe import compute_epe
from openlithohub.benchmark.metrics.pvband import compute_pvband

predicted = binary_erosion(binary_dilation(target, radius=2), radius=1)

epe = compute_epe(predicted, target, pixel_size_nm=1.0)
pvb = compute_pvband(predicted, pixel_size_nm=1.0)
print("EPE :", epe)
print("PV  :", pvb)

## 4. Paper-ready figure

`openlithohub.vis.plot_contours` produces a single-panel target/predicted overlay using the IEEE column-width style. Pass `save_path='fig.pdf'` for a vector figure suitable for a manuscript.

In [ ]:
from openlithohub.vis import plot_contours

fig = plot_contours(
    target,
    predicted,
    pixel_size_nm=1.0,
    title="OpenLithoHub quickstart",
    style="ieee",
)
fig

## Next steps

- Wire your own model to the `LithographyModel` interface — see the docs at <https://docs.openlithohub.com>.
- Swap the dummy layout for `LithoBenchDataset` once you've installed the `[data]` extra.
- Export to OASIS and a Calibre/IC Validator deck via `openlithohub.workflow.emit_bridge_bundle` (requires the `[workflow]` extra).